In [1]:
import pandas as pd
import matplotlib.pyplot as plt
print("pandas", pd.__version__)

Matplotlib is building the font cache; this may take a moment.


pandas 3.1.0


In [3]:
import sys
print(sys.executable)


/Users/terry/Documents/ni-ireland-inactivity/.venv/bin/python


In [6]:
import os
print(os.listdir("../data/raw"))


['women-in-NI-headline-tables-2023.XLSX', 'labour-market-structure-2025.xlsx', '.gitkeep', 'industry-2025.xlsx', 'Labour Force Survey - Tables for Local Government Districts 2009 to 2025.xlsx']


In [8]:
path = "../data/raw/Labour Force Survey - Tables for Local Government Districts 2009 to 2025.xlsx"
sheets = pd.read_excel(path, sheet_name=None, header=None)

rows = []
for name, df in sheets.items():
    if not name.isdigit():
        continue
    ni_rows = df[df[0].str.strip() == "Northern Ireland"]
    activity = ni_rows.iloc[-1, 2]
    rows.append({"year": int(name), "inactivity": 100 - activity})

ni = pd.DataFrame(rows).sort_values("year")
print(ni)

    year  inactivity
16  2009        30.1
15  2010        29.1
14  2011        27.6
13  2012        27.6
12  2013        27.5
11  2014        27.6
10  2015        27.1
9   2016        26.0
8   2017        27.6
7   2018        27.2
6   2019        26.2
5   2020        28.0
4   2021        27.0
3   2022        26.3
2   2023        24.7
1   2024        24.3
0   2025        24.1


In [10]:
path_ie = "../data/raw/LRAC64TTIEA156N.csv"
ie = pd.read_csv(path_ie)
print(ie.head())

  observation_date  LRAC64TTIEA156N
0       2000-01-01           68.175
1       2001-01-01           68.550
2       2002-01-01           68.550
3       2003-01-01           68.825
4       2004-01-01           69.500


In [11]:
ie["year"] = ie["observation_date"].str[:4].astype(int)
ie["inactivity"] = 100 - ie["LRAC64TTIEA156N"]
ie = ie[(ie["year"] >= 2009) & (ie["year"] <= 2025)]
ie = ie[["year", "inactivity"]]

both = ni.merge(ie, on="year", suffixes=("_ni", "_ie"))
both["gap"] = both["inactivity_ni"] - both["inactivity_ie"]
print(both.round(1))

    year  inactivity_ni  inactivity_ie  gap
0   2009           30.1           27.0  3.1
1   2010           29.1           28.4  0.6
2   2011           27.6           28.8 -1.2
3   2012           27.6           28.9 -1.3
4   2013           27.5           28.2 -0.7
5   2014           27.6           28.2 -0.6
6   2015           27.1           28.0 -0.9
7   2016           26.0           27.4 -1.4
8   2017           27.6           27.4  0.2
9   2018           27.2           27.2 -0.0
10  2019           26.2           26.8 -0.6
11  2020           28.0           29.2 -1.2
12  2021           27.0           25.5  1.5
13  2022           26.3           23.2  3.0
14  2023           24.7           22.5  2.2
15  2024           24.3           22.1  2.2
16  2025           24.1           21.6  2.5


In [13]:
ni.to_csv("../data/clean/ni_inactivity_2009_2025.csv", index=False)
ie.to_csv("../data/clean/ie_inactivity_2009_2025.csv", index=False)

In [14]:
both.to_csv("../data/clean/ni_ie_inactivity_2009_2025.csv", index=False)